# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadSulaiman08/FLYRANK-ML-INTERNSHIP/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane is a **ranking** problem: put the pages most likely to be declining at the top of an editor's list. So I train classifiers and use their **probability as a ranking score**, judged at Precision@K.

I compare, from simplest to most complex: **Logistic Regression** (readable, linear), a **depth-4 Decision Tree** (printable rules), **Random Forest** and **Gradient Boosting** (non-linear, combine many weak signals). Complexity has to earn its place: a complex model only "wins" if it clearly beats the simple ones and the rule baselines on unseen clients.

Features are **pre-decision signals only** (visibility, clicks, sessions, position, CTR, age, freshness, length, engagement, intent/type). `trend_direction` and `trend_pct` are excluded because the label is built from them (leakage). The code checks this.## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os, subprocess
import numpy as np, pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(CSV):
    for up in ("..", "../.."):
        if os.path.exists(os.path.join(up, CSV)):
            os.chdir(up); break
    else:
        if not os.path.isdir("starter"):
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "starter"], check=True)
        os.chdir("starter")

RANDOM_STATE = 42
df = pd.read_csv(CSV)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
df["y"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Pre-decision signals only (same idea as the reference pipeline's feature list)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df["log_" + c] = np.log1p(df[c].clip(lower=0))
NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
       "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent", "position_tier"]

LEAKY = {"trend_direction", "trend_pct", "y"}
assert not LEAKY & set(NUM + CAT), "leakage: label-derived column in features"

X_num = df[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
X = pd.concat([X_num, pd.get_dummies(df[CAT].fillna("missing").astype(str), drop_first=False).astype(int)], axis=1)
y, groups = df["y"].values, df["client_id"].values
print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | {X.shape[1]} features | base rate {y.mean():.3f}")
print("Leakage guard passed: trend_direction / trend_pct are NOT features.")
import sklearn; print("scikit-learn", sklearn.__version__, "| random_state =", RANDOM_STATE)

30,000 pages | 32 clients | 35 features | base rate 0.542
Leakage guard passed: trend_direction / trend_pct are NOT features.
scikit-learn 1.6.1 | random_state = 42


## 2. Split design

**5-fold GroupKFold, grouped by `client_id`.** Every test fold contains only clients the model never saw in training (the code prints "shared clients: 0" for each fold).

Why this is honest for my question: an editor would use the ranking on a *new* site. If pages from the same client sit in both train and test, the model can memorise that client's style and look better than it really is. Five folds instead of one split also show how much the result swings between groups of clients (the P@50 std column). One fold holds a single large client, which is part of why results vary.

Limit: this split is not time-aware. The starter label comes from the current trend window, so this checks "generalises to new clients", not "predicts the future".## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X, y, groups))
for i, (tr, te) in enumerate(folds, 1):
    overlap = set(groups[tr]) & set(groups[te])
    print(f"fold {i}: train {len(tr):>6,} pages / {len(set(groups[tr]))} clients | "
          f"test {len(te):>5,} pages / {len(set(groups[te]))} clients | shared clients: {len(overlap)}")

fold 1: train 22,992 pages / 31 clients | test 7,008 pages / 1 clients | shared clients: 0
fold 2: train 24,269 pages / 25 clients | test 5,731 pages / 7 clients | shared clients: 0
fold 3: train 24,247 pages / 24 clients | test 5,753 pages / 8 clients | shared clients: 0
fold 4: train 24,245 pages / 24 clients | test 5,755 pages / 8 clients | shared clients: 0
fold 5: train 24,247 pages / 24 clients | test 5,753 pages / 8 clients | shared clients: 0


## 3. Train + compare vs my baseline

Same data, same 5 client-grouped folds, same metrics for every method. Two baselines need no training: the **hand rule** from notebook 02 (stale × visible) and the **weighted rule** from the reference pipeline.

**Observed (mean over 5 folds):**
- Gradient Boosting ranked best: P@50 ≈ 0.84, vs 0.60 for the hand rule, 0.46 for the weighted rule, and 0.54 for random order. It was also the most stable across folds (lowest P@50 std).
- Random Forest (≈ 0.80) and plain Logistic Regression (≈ 0.78) were close behind. Most of the gain comes from *combining signals at all*, not from model complexity.
- The depth-4 tree (≈ 0.60) only matched the hand rule.
- ROC AUC stays modest (≈ 0.69 at best), so the models are good at the top of the list but far from perfect overall.

**My pick:** Gradient Boosting for the ranked queue, with Logistic Regression as the readable fallback, since the gap between them is small.## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

def p_at_k(y_true, score, k):
    order = np.argsort(-np.asarray(score), kind="stable")
    return np.asarray(y_true)[order[:k]].mean()

def pct(s):
    return pd.Series(s).rank(pct=True).values

# Baselines (no training): the notebook-02 hand rule and the reference pipeline's weighted rule
vis = pct(np.log1p(df["impressions_90d"]))
fresh = pct(df["days_since_last_update"])
pos = df["avg_position"].clip(1, 50)
pos_opp = (1 - (pos - pos.min()) / (pos.max() - pos.min())) * vis * (df["avg_position"] > 0)
depth = (1 - pct(df["word_count"].fillna(0))) * vis
BASELINES = {
    "Hand rule (stale x visible)": ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)) * df["impressions_90d"],
    "Weighted rule (pipeline baseline)": 0.40 * vis + 0.30 * fresh + 0.25 * pos_opp + 0.05 * depth,
}
MODELS = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Decision Tree (depth 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=RANDOM_STATE),
}

rows, oof = [], {}
for name in list(BASELINES) + list(MODELS):
    oof[name] = np.zeros(len(df))
for f, (tr, te) in enumerate(folds):
    for name, s in BASELINES.items():
        oof[name][te] = np.asarray(s, dtype=float)[te]
    for name, m in MODELS.items():
        m.fit(X.iloc[tr], y[tr])
        oof[name][te] = m.predict_proba(X.iloc[te])[:, 1]
    for name in oof:
        sc = oof[name][te]
        rows.append({"method": name, "fold": f + 1, "P@20": p_at_k(y[te], sc, 20), "P@50": p_at_k(y[te], sc, 50),
                     "P@100": p_at_k(y[te], sc, 100), "ROC AUC": roc_auc_score(y[te], sc),
                     "Avg precision": average_precision_score(y[te], sc)})

res = pd.DataFrame(rows)
table = res.groupby("method", sort=False)[["P@20", "P@50", "P@100", "ROC AUC", "Avg precision"]].mean()
table["P@50 std"] = res.groupby("method", sort=False)["P@50"].std()
table.loc["Random order (base rate)"] = [y.mean(), y.mean(), y.mean(), 0.5, y.mean(), 0.0]
print("Mean over 5 client-grouped folds (each test fold = clients never seen in training):")
table.round(3)

Mean over 5 client-grouped folds (each test fold = clients never seen in training):


,P@20,P@50,P@100,ROC AUC,Avg precision,P@50 std
method,,,,,,
Hand rule (stale x visible),0.640,0.600,0.572,0.500,0.545,0.166
Weighted rule (pipeline baseline),0.420,0.464,0.466,0.599,0.595,0.173
Logistic Regression,0.780,0.776,0.776,0.667,0.677,0.153
Decision Tree (depth 4),0.560,0.600,0.630,0.638,0.631,0.110
Random Forest,0.750,0.784,0.776,0.677,0.687,0.124
Gradient Boosting,0.820,0.768,0.794,0.685,0.692,0.129
Random order (base rate),0.542,0.542,0.542,0.500,0.542,0.000


In [4]:
from sklearn.inspection import permutation_importance

tr, te = folds[0]
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=RANDOM_STATE).fit(X.iloc[tr], y[tr])
perm = permutation_importance(rf, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 8 features by permutation importance (drop in ROC AUC when shuffled, held-out clients):")
print(imp.head(8).round(4).to_string())

test = df.iloc[te].copy()
test["score"] = rf.predict_proba(X.iloc[te])[:, 1]
top = test.sort_values("score", ascending=False).head(50)
print(f"\nTop-50 on held-out fold: {top['y'].sum()} declining, {(1 - top['y']).sum()} false alarms")
print("\nFalse alarms in the top 50 — what trend did they really have?")
print(top.loc[top["y"] == 0, "trend_direction"].value_counts().to_string())

print("\nModel score vs. content age (held-out fold): share declining and mean score per age group")
test["age_group"] = pd.cut(test["content_age_days"], [0, 180, 365, 730, 10**5], labels=["<6m", "6-12m", "1-2y", "2y+"])
print(test.groupby("age_group", observed=True).agg(pages=("y", "size"), declining=("y", "mean"), mean_score=("score", "mean")).round(3).to_string())

missed = test[(test["y"] == 1)].sort_values("score").head(3)
print("\n3 declining pages the model ranked LOWEST (hard misses):")
print(missed[["impressions_90d", "ctr", "avg_position", "content_age_days", "days_since_last_update", "word_count", "score"]].round(3).to_string(index=False))

Top 8 features by permutation importance (drop in ROC AUC when shuffled, held-out clients):
avg_position              0.0563
content_age_days          0.0272
position_tier_page_1      0.0095
position_tier_page_3_5    0.0062
days_with_impressions     0.0031
position_tier_deep        0.0027
position_tier_top_3       0.0025
word_count                0.0013

Top-50 on held-out fold: 41 declining, 9 false alarms

False alarms in the top 50 — what trend did they really have?
trend_direction
up        6
stable    3

Model score vs. content age (held-out fold): share declining and mean score per age group
           pages  declining  mean_score
age_group                              
<6m         1431      0.628       0.587
6-12m       2570      0.498       0.548
1-2y        3007      0.418       0.451

3 declining pages the model ranked LOWEST (hard misses):
 impressions_90d  ctr  avg_position  content_age_days  days_since_last_update  word_count  score
             286 0.35          53.6     

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.